In [1]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.io import imread
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/home/jie/Documents/NextFace_custom')
DECA_ROOT = PROJECT_ROOT / 'ext' / 'deca'
sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_ROOT = Path('/home/jie/Downloads/vggface2_train/train')
FIRST_PASS_KPT_ROOT = Path('/home/jie/Downloads/vggface2_train/train_annotated_fan')
KPT_ROOT = FIRST_PASS_KPT_ROOT

CLEAN_LIST_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_list.npy')
CLEAN_METADATA_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_metadata.npz')
ACCEPTED_NAMES_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_names.txt')

FIRST_PASS_METADATA_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_creation_metadata.npz'
FIRST_PASS_SAVED_NAMES_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_creation_saved_names.txt'
FIRST_PASS_FAILURE_LOG_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_failures.txt'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('IMAGE_ROOT =', IMAGE_ROOT, IMAGE_ROOT.exists())
print('FIRST_PASS_KPT_ROOT =', FIRST_PASS_KPT_ROOT, FIRST_PASS_KPT_ROOT.exists())
print('CLEAN_LIST_PATH =', CLEAN_LIST_PATH)
print('FIRST_PASS_METADATA_PATH =', FIRST_PASS_METADATA_PATH, FIRST_PASS_METADATA_PATH.exists())


PROJECT_ROOT = /home/jie/Documents/NextFace_custom
IMAGE_ROOT = /home/jie/Downloads/vggface2_train/train True
FIRST_PASS_KPT_ROOT = /home/jie/Downloads/vggface2_train/train_annotated_fan True
CLEAN_LIST_PATH = /home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_list.npy
FIRST_PASS_METADATA_PATH = /home/jie/Downloads/vggface2_train/fan_landmark_creation_metadata.npz True


/home/jie/Documents/NextFace_custom/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from landmarks.fan import LandmarksDetectorFAN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mask = torch.arange(68, dtype=torch.long)
fan = LandmarksDetectorFAN(mask=mask, device=device)

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device =', device)

torch version: 2.11.0+cu130
cuda available: True
device = cuda


In [3]:
image_paths = sorted(
    list(IMAGE_ROOT.glob('*/*.jpg'))
    + list(IMAGE_ROOT.glob('*/*.jpeg'))
    + list(IMAGE_ROOT.glob('*/*.png'))
)
image_path_by_name = {
    image_path.relative_to(IMAGE_ROOT).with_suffix('').as_posix(): image_path
    for image_path in image_paths
}

print('num images:', len(image_paths))
print('first image:', image_paths[0] if image_paths else None)


def image_name_for_path(image_path):
    return Path(image_path).relative_to(IMAGE_ROOT).with_suffix('').as_posix()


def kpt_path_for_image(image_path):
    rel = Path(image_path).relative_to(IMAGE_ROOT)
    return (FIRST_PASS_KPT_ROOT / rel).with_suffix('.npy')


if image_paths:
    print('first cached first-pass landmarks:', kpt_path_for_image(image_paths[0]))


num images: 3141890
first image: /home/jie/Downloads/vggface2_train/train/n000002/0001_01.jpg
first cached first-pass landmarks: /home/jie/Downloads/vggface2_train/train_annotated_fan/n000002/0001_01.npy


In [4]:
def add_unique(name_list, name_set, name):
    if name not in name_set:
        name_list.append(name)
        name_set.add(name)

def read_image_rgb_uint8(image_path):
    image = imread(image_path)
    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=2)
    if image.shape[2] == 4:
        image = image[..., :3]
    if image.dtype == np.uint8:
        return image
    return np.clip(image, 0, 255).astype(np.uint8)

In [5]:
# import time
# # CPU
# for image_path in tqdm(image_paths_to_process[:1000], desc='profiling FAN'):
#     name = image_name_for_path(image_path)
#     output_path = annotated_landmark_path_for_image(image_path)

#     t0 = time.perf_counter()
#     image = read_image_rgb_uint8(image_path)
#     t1 = time.perf_counter()

#     if torch.cuda.is_available():
#         torch.cuda.synchronize()
#     with torch.no_grad():
#         landmarks = fan(image)
#     if torch.cuda.is_available():
#         torch.cuda.synchronize()
#     t2 = time.perf_counter()

#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     np.save(output_path, np.asarray(landmarks, dtype=np.float32)[:, :2])
#     t3 = time.perf_counter()

#     print('read:', t1 - t0, 'fan:', t2 - t1, 'save:', t3 - t2)
#     break


In [8]:
landmark_parent_dirs = sorted({
    annotated_landmark_path_for_image(image_path).parent
    for image_path in image_paths_to_process
})

for parent_dir in tqdm(landmark_parent_dirs, desc='creating landmark folders'):
    parent_dir.mkdir(parents=True, exist_ok=True)

print('landmark folders ready:', len(landmark_parent_dirs))


KeyboardInterrupt: 

In [13]:
ANNOTATED_LANDMARKS_ROOT = Path(r'/home/jie/Downloads/vggface2_train/train_annotated_fan')

def annotated_landmark_path_for_image(image_path):
    name = image_name_for_path(image_path)
    return (ANNOTATED_LANDMARKS_ROOT / name).with_suffix('.npy')

first_missing_index = None
first_missing_image_path = None
first_missing_name = None

for i, image_path in enumerate(tqdm(image_paths, desc='finding first missing landmark')):
    output_path = annotated_landmark_path_for_image(image_path)

    if not output_path.is_file():
        first_missing_index = i
        first_missing_image_path = image_path
        first_missing_name = image_name_for_path(image_path)
        break

if first_missing_index is None:
    image_paths_to_process = []
else:
    image_paths_to_process = image_paths[first_missing_index:]

print('first missing index:', first_missing_index)
print('first missing name:', first_missing_name)
print('images left to process:', len(image_paths_to_process))


finding first missing landmark:   1%|▏         | 40957/3141890 [00:00<01:14, 41408.47it/s]

first missing index: 40957
first missing name: n000120/0399_01
images left to process: 3100933


In [ ]:
for image_path in tqdm(image_paths_to_process, desc='saving FAN landmarks'):
    name = image_name_for_path(image_path)
    output_path = annotated_landmark_path_for_image(image_path)

    image = read_image_rgb_uint8(image_path)
    with torch.no_grad():
        landmarks = fan(image)
    if landmarks is None:
        continue

    landmarks = np.asarray(landmarks, dtype=np.float32)
    if landmarks.ndim != 2 or landmarks.shape[1] < 2:
        print(f'skipping {name}: unexpected landmark shape {landmarks.shape}')
        continue

    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(output_path, landmarks[:, :2])


print('queued for FAN:', len(image_paths_to_process))


saving FAN landmarks:   0%|          | 0/3100933 [00:00<?, ?it/s]/home/jie/Documents/NextFace_custom/.venv/lib/python3.14/site-packages/face_alignment/api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")
saving FAN landmarks:   0%|          | 24/3100933 [00:06<517:14:01,  1.67it/s]

found multiple subjects in image. extracting landmarks for first subject only...


saving FAN landmarks:   0%|          | 29/3100933 [00:10<643:45:03,  1.34it/s]

found multiple subjects in image. extracting landmarks for first subject only...


saving FAN landmarks:   0%|          | 31/3100933 [00:12<739:26:09,  1.16it/s]

...

saving FAN landmarks:   0%|          | 49/3100933 [00:23<519:09:22,  1.66it/s]

found multiple subjects in image. extracting landmarks for first subject only...


saving FAN landmarks:   0%|          | 49/3100933 [00:23<410:26:10,  2.10it/s]


KeyboardInterrupt: 

: 

In [10]:
image_path

PosixPath('/home/jie/Downloads/vggface2_train/train/n000120/0399_01.jpg')

In [ ]:
for image_path in tqdm(candidate_image_paths[:100], desc='saving FAN landmarks'):

[WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0001_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0002_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0003_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0004_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0005_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0006_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0007_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0008_01.jpg'),
 WindowsPath('C:/Users/kevin/Documents/deca_dataset/VGG-Face2/data/vggface2_train/train/n000002/0009_01.jpg'),
 

In [ ]:
if landmark_path is None or not landmark_path.is_file():
    raise RuntimeError('Run the FAN landmark save cell first; no saved landmark file is available to visualize.')

overlay_image = imread(first_image_path)
if overlay_image.ndim == 2:
    overlay_image = np.repeat(overlay_image[..., None], 3, axis=2)
if overlay_image.shape[2] == 4:
    overlay_image = overlay_image[..., :3]

loaded_landmarks = np.load(landmark_path)

plt.figure(figsize=(7, 7))
plt.imshow(overlay_image)
plt.scatter(
    loaded_landmarks[:, 0],
    loaded_landmarks[:, 1],
    s=12,
    c='lime',
    edgecolors='black',
    linewidths=0.4,
)
plt.title(landmark_path.name)
plt.axis('off')
plt.show()